# 04 — Corridor Analysis: Buffering, Clipping, and Exceedance Detection

## What this notebook does

This notebook applies the corridor analysis pipeline to the CHM from Notebook 03:

1. Load the transmission line centerline shapefile
2. Buffer the centerline by the corridor width (default: ~15m / 50ft)
3. Clip the full CHM to the corridor buffer extent
4. Apply a height threshold to detect trees exceeding the clearance standard
5. Export flagged tree detections and summary statistics as final output artifacts

This notebook produces all three final deliverables for this project.

---

## Inputs expected

- `data/raw/<corridor_centerline>.shp` or `.gpkg` — transmission line
  centerline in a projected CRS (units: meters). Source: CA Energy Commission GIS.
- `data/processed/chm.tif` — full-extent CHM GeoTIFF from Notebook 03.
- CHM and centerline must share the same CRS before any spatial operation.

## Outputs produced

- `outputs/rasters/chm_clipped.tif` — CHM raster clipped to the corridor
  buffer extent. Produced by `clip_chm_to_corridor()`.
- `outputs/vector/flagged_trees.gpkg` — GeoPackage of individual tree
  detections exceeding the height threshold, with `height_m` attribute.
  Produced by `threshold_exceedance()`.
- `outputs/tables/flagged_tree_summary.csv` — summary table of flagged
  tree count and mean height by corridor segment. Produced by aggregating
  the flagged_trees GeoDataFrame.

## src/ functions called

- `src.corridor.load_corridor_centerline(filepath)`
- `src.corridor.buffer_corridor(gdf, buffer_distance_m)`
- `src.corridor.clip_chm_to_corridor(chm_path, corridor_gdf, output_path)`
- `src.corridor.threshold_exceedance(chm_clipped_path, height_threshold_m)`

---

## Key decisions to make before writing code

- **Buffer distance:** The default 15m (~50ft) is illustrative and derived
  from typical 200–500kV right-of-way standards under NERC FAC-003-4.
  Verify the applicable voltage class for the study corridor and consult
  FAC-003-4 Table 1 before using this value operationally.
- **Height threshold:** The default 4.57m (15ft) is illustrative. Actual
  utility vegetation management standards vary by voltage class and
  jurisdiction. Verify before interpreting exceedance results.
- **Segment definition:** The summary CSV requires aggregation by corridor
  segment. Decide how segments will be defined (e.g., by original feature
  ID in the centerline shapefile, or by distance intervals).
- **CRS alignment:** Confirm the corridor centerline CRS matches the CHM CRS
  before buffering or clipping. Reproject one or both if needed.
